**Bigger-font variant (2026-08 supervisor feedback item 2).** Copied from the original
simple-plot notebook, not edited in place — same convention as
`100_versions_pie_plot_simple_bigfont.ipynb` and the original-vs-simple-plot split before it.
Reads from the bigfont chart source (`100_pie_charts_simple_bigfont/`) and writes to a separate
`*_bigfont` post/output tree throughout, so nothing here collides with the simple-plot pilot's
own data or results.

This pilot's stimuli (2,400 HTML + 2,400 PNG) are already generated and tracked, so none of the
cells below need to run again for a normal clone — they document how the data was built, and are
the reference if it is ever extended or regenerated. No `.py` script covers this chart pool.

`likes_only`/`uniform` were never generated for this pilot — only `realistic` and
`likes_only_noise`, matching the two conditions kept for the main study (2026-09-17).

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path().resolve().parent

## Baseline (zero engagement)

Both variants, one loop — matches the pattern used everywhere else in this codebase (`generate_profile_posts.py`'s `VARIANTS` dict).

In [ ]:
from post_generator_all_visible_emojis import generate_facebook_post, display_facebook_post

PROFILE_NAME = "Remy Ashford"
POST_TIME = "Today at 2:43 PM"
VERIFIED = False
PROFILE_IMAGE_PATH = None

REACTIONS = {"like": 0, "love": 0, "haha": 0, "wow": 0, "sad": 0, "angry": 0}
COMMENT_COUNT = 0
SHARE_COUNT = 0

VARIANTS = {
    "correct":   "The Spotify 2026 music genres distribution has just been released. "
                 "Looks like Pop was more popular than Latin this year!",
    "incorrect": "The Spotify 2026 music genres distribution has just been released. "
                 "Looks like Latin was more popular than Pop this year!",
}
SUFFIX = {"correct": "c", "incorrect": "i"}

for variant, post_text in VARIANTS.items():
    for i in range(1, 101):
        POST_IMAGE_PATH = os.path.join(BASE_DIR, f"spotify_pie_plot/100_pie_charts_simple_bigfont/spotify_genre_pie_chart_simple_{i:03d}.png")
        output_file = os.path.join(BASE_DIR, f"spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/{variant}/html/{i:03d}_remy_ashford_{SUFFIX[variant]}.html")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        html_content = generate_facebook_post(
            profile_name=PROFILE_NAME, post_text=post_text, post_time=POST_TIME,
            reactions=REACTIONS, comment_count=COMMENT_COUNT, share_count=SHARE_COUNT,
            post_image_path=POST_IMAGE_PATH, output_file=output_file,
            verified=VERIFIED, profile_image_path=PROFILE_IMAGE_PATH,
        )
    print(f"  -> '{variant}': done")


## Metrics: realistic

Log-weighted, shuffled reaction shares (seed=post number). See the *angry always gets the smallest weight* note in the main study's own generator for why negative reactions are structurally underrepresented here — a known, named limitation, not a bug, and not re-derived per pilot.

In [ ]:
import importlib
import math
import random
import os
import post_generator_all_visible_emojis
importlib.reload(post_generator_all_visible_emojis)
from post_generator_all_visible_emojis import generate_facebook_post, display_facebook_post

PROFILE_NAME = "Remy Ashford"
POST_TIME = "Today at 2:43 PM"
VERIFIED = False
PROFILE_IMAGE_PATH = None
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000] # log scale
VARIANTS = {
    "incorrect": "The Spotify 2026 music genres distribution has just been released. Looks like Latin was more popular than Pop this year!",
    "correct": "The Spotify 2026 music genres distribution has just been released. Looks like Pop was more popular than Latin this year!",
}
SUFFIX = {"incorrect": "i", "correct": "c"}

REACTION_TYPES = ["like", "love", "haha", "wow", "sad", "angry"]

def make_log_reactions(scale_value, jitter=0.10, seed=None):
    rng = random.Random(seed)
    n = len(REACTION_TYPES)
    
    log_weights = [math.log(n + 1 - i) for i in range(n)]
    total_weight = sum(log_weights)
    shares = [w / total_weight for w in log_weights]
    
    # Shuffle the shares, not the reaction types
    rng.shuffle(shares)
    
    reactions = {}
    for emoji, share in zip(REACTION_TYPES, shares):
        base = scale_value * share
        factor = 1 + rng.uniform(-jitter, jitter)
        reactions[emoji] = max(1, round(base * factor))
    return reactions

for scale_value in REACTION_VALUES:
    for variant, post_text in VARIANTS.items():
        for i in range(1, 101):
            REACTIONS = make_log_reactions(scale_value, jitter=0.10, seed=i)
            COMMENT_COUNT = max(1, round(scale_value * 0.08))
            SHARE_COUNT   = max(1, round(scale_value * 0.04))

            POST_IMAGE_PATH = os.path.join(BASE_DIR, f"spotify_pie_plot/100_pie_charts_simple_bigfont/spotify_genre_pie_chart_simple_{i:03d}.png")
            output_file = os.path.join(BASE_DIR, f"spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/{variant}/html/realistic/{scale_value}/{i:03d}_remy_ashford_{SUFFIX[variant]}.html")

            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            html_content = generate_facebook_post(
                profile_name=PROFILE_NAME,
                post_text=post_text,
                post_time=POST_TIME,
                reactions=REACTIONS,
                comment_count=COMMENT_COUNT,
                share_count=SHARE_COUNT,
                post_image_path=POST_IMAGE_PATH,
                output_file=output_file,
                verified=VERIFIED,
                profile_image_path=PROFILE_IMAGE_PATH,
            )
    print(f"✅ Generated scale value: {scale_value}")
print("\n✅ All metric variants complete.")

## Metrics: likes_only_noise

In [ ]:
# likes_only_noise (bigfont) -- REUSES THE MAIN STUDY'S ENGAGEMENT VALUES.
#
# Rewritten 2026-08-20. This cell previously generated its own jitter with an UNSEEDED
# random.choice, so the bigfont pilot carried different engagement numbers from the main
# study it is compared against -- and different numbers again on every re-run. Since the
# point of this pilot is to isolate FONT SIZE, every other property should be held fixed.
#
# The main study's values are not recomputable (they were unseeded too), so they are read
# back out of its generated posts. The raw count is taken from the page's JS
# (`let currentLikes = N`), not the rendered bubble text, because the bubble is abbreviated
# ("1.1K") and parsing it would be lossy.
#
# Verified: the main study used the SAME value for the correct and incorrect variants of a
# given (scale, post) -- 600/600 identical -- matching this loop's original structure.

import os, re, csv, importlib
import post_generator_all_visible_emojis
importlib.reload(post_generator_all_visible_emojis)
from post_generator_all_visible_emojis import generate_facebook_post


PROFILE_NAME = "Remy Ashford"
POST_TIME = "Today at 2:43 PM"
VERIFIED = False
PROFILE_IMAGE_PATH = None
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]
VARIANTS = {
    "incorrect": "The Spotify 2026 music genres distribution has just been released. Looks like Latin was more popular than Pop this year!",
    "correct": "The Spotify 2026 music genres distribution has just been released. Looks like Pop was more popular than Latin this year!",
}
SUFFIX = {"incorrect": "i", "correct": "c"}

# Overwrite rather than skip. The previous stimuli were generated while the shared post
# generator was broken (commit 909a2658e: zeros dropped, reactions re-sorted, "1.0K"
# formatting), so any file already on disk is the one we are trying to replace.
FORCE_REGENERATE = True

MAIN_NOISE = os.path.join(BASE_DIR, "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford")
CURRENT_LIKES = re.compile(r"let currentLikes = (\d+)")

def main_study_likes(scale_value):
    """{post_number: like_count} as used by the main study at this scale."""
    out = {}
    for i in range(1, 101):
        f = os.path.join(MAIN_NOISE, f"correct/html/likes_only_noise/{scale_value}/{i:03d}_remy_ashford_c.html")
        if not os.path.exists(f):
            continue
        with open(f, encoding="utf-8", errors="ignore") as fh:
            m = CURRENT_LIKES.search(fh.read())
        if m:
            out[i] = int(m.group(1))
    return out

csv_log_path = os.path.join(BASE_DIR, "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/html/likes_only_noise/jitter_assignment_log_likes_only_noise.csv")
os.makedirs(os.path.dirname(csv_log_path), exist_ok=True)
with open(csv_log_path, "w", newline="") as log_file:
    csv.writer(log_file).writerow(["Image_Index", "Base_Scale", "Assigned_Likes", "Actual_Variance", "Source"])

for scale_value in REACTION_VALUES:
    values = main_study_likes(scale_value)
    if len(values) != 100:
        raise RuntimeError(
            f"scale {scale_value}: found {len(values)}/100 main-study values under {MAIN_NOISE}. "
            f"Refusing to fall back to fresh randomisation -- that is exactly the confound "
            f"this cell was rewritten to remove."
        )

    with open(csv_log_path, "a", newline="") as log_file:
        w = csv.writer(log_file)
        for i in range(1, 101):
            v = values[i]
            w.writerow([f"{i:03d}", scale_value, v, f"{((v - scale_value) / scale_value) * 100:+.1f}%", "main-study"])

    for variant, post_text in VARIANTS.items():
        generated = 0
        for i in range(1, 101):
            output_file = os.path.join(
                BASE_DIR,
                f"spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/{variant}/html/likes_only_noise/{scale_value}/{i:03d}_remy_ashford_{SUFFIX[variant]}.html",
            )
            if os.path.exists(output_file) and not FORCE_REGENERATE:
                continue
            REACTIONS = {"like": values[i], "love": 0, "haha": 0, "wow": 0, "sad": 0, "angry": 0}
            POST_IMAGE_PATH = os.path.join(
                BASE_DIR,
                f"spotify_pie_plot/100_pie_charts_simple_bigfont/spotify_genre_pie_chart_simple_{i:03d}.png",
            )
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            generate_facebook_post(
                profile_name=PROFILE_NAME, post_text=post_text, post_time=POST_TIME,
                reactions=REACTIONS, comment_count=0, share_count=0,
                post_image_path=POST_IMAGE_PATH, output_file=output_file,
                verified=VERIFIED, profile_image_path=PROFILE_IMAGE_PATH,
            )
            generated += 1
        print(f"  -> '{variant}': generated {generated}")
    print(f"Finished scale {scale_value} (values reused from the main study).")

print("\nAll likes_only_noise variants regenerated with the main study's engagement values.")
